# MFFT Training — Step by Step
**Multi-Frequency Fusion Transformer for AI Image Detection**

In [1]:
# Cell 1: Imports & Setup
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

# Fix project root
PROJECT_ROOT = Path(os.path.abspath('')).parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
print(f'Project root: {PROJECT_ROOT}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')


Project root: d:\Research\ai-image-detection-research
Torch: 2.12.1+cpu, CUDA: False


In [2]:
# Cell 2: Load Dataset
from src.dataset import AIDetectionDataset, ImageTransform, create_dataloaders
from src.config import Config

cfg = Config()
cfg.training.epochs = 50
cfg.training.model_variant = 'base'
cfg.training.image_size = 384
cfg.training.batch_size = 8
cfg.training.mixed_precision = False
cfg.training.num_workers = 0
cfg.training.val_check_interval = 50
cfg.training.gradient_accumulation_steps = 1
cfg.dataset.val_split = 0.1
cfg.dataset.test_split = 0.1

# Load full dataset (uses clean_metadata.csv by default now)
full_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT),
    metadata_paths=cfg.dataset.metadata_paths,
    transform=None,
    is_train=True,
    size=cfg.training.image_size,
    undersample=True,
)

print(f'Total samples: {len(full_dataset)}')
print(f'Real: {sum(1 for _, l in full_dataset.samples if l==0)}')
print(f'AI:   {sum(1 for _, l in full_dataset.samples if l==1)}')

Dataset loaded: 50000 samples
  Real: 25000, AI: 25000, Total: 50000
Total samples: 50000
Real: 25000
AI:   25000


In [3]:
# Cell 3: Split into Train/Val/Test
from sklearn.model_selection import train_test_split

labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(
    indices, test_size=cfg.dataset.val_split + cfg.dataset.test_split,
    stratify=labels, random_state=42,
)

temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=cfg.dataset.test_split / (cfg.dataset.val_split + cfg.dataset.test_split),
    stratify=temp_labels, random_state=42,
)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

# Build datasets with transforms
train_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=True),
    is_train=True, size=cfg.training.image_size, undersample=False,
)
train_dataset.samples = [full_dataset.samples[i] for i in train_idx]

val_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
val_dataset.samples = [full_dataset.samples[i] for i in val_idx]

test_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
test_dataset.samples = [full_dataset.samples[i] for i in test_idx]

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

Train: 40000, Val: 5000, Test: 5000
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Train batches: 5000, Val batches: 625, Test batches: 625


In [4]:
# Cell 4: Build Model
from src.model import build_mfft, count_parameters, MFFTWithExplainability

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = build_mfft('base')
model = model.to(device)
print(f'Parameters: {count_parameters(model):,}')

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
print('Model ready')

Device: cpu
Parameters: 866,498
Model ready


In [ ]:
# Cell 4b: Baseline Models Overview
from src.baselines import (
    SimpleCNN, LightViT, count_parameters,
    resnet18, resnet50, efficientnet_b0, vit_b_16, swin_t,
    CLIPBaseline,
)

baseline_models = {
    'SimpleCNN': lambda: SimpleCNN(),
    'LightViT': lambda: LightViT(depth=4, num_heads=4, embed_dim=192),
    'ResNet-18': lambda: resnet18(),
    'ResNet-50': lambda: resnet50(),
    'EfficientNet-B0': lambda: efficientnet_b0(),
    'ViT-B/16': lambda: vit_b_16(img_size=cfg.training.image_size),
    'Swin-T': lambda: swin_t(),
    'CLIP': lambda: CLIPBaseline(),
}

x = torch.randn(2, 3, cfg.training.image_size, cfg.training.image_size)
print(f"{'Model':<20} {'Params':>10} {'Output':>10}")
print('-' * 42)
for name, fn in baseline_models.items():
    m = fn()
    p = count_parameters(m)
    o = list(m(x).shape)
    print(f'{name:<20} {p:>10,}  {str(o):>10}')
print()
print('To train a specific baseline, replace model in Cell 4 with:')
print("  model = resnet50().to(device)")
print('Then run Cells 5-9 as-is.')


In [5]:
# Cell 5: Train 1 Epoch (watch progress)
model.train()
total_loss = 0
correct = 0
total = 0
start = time.time()

pbar = tqdm(train_loader, desc='Training')
for batch_idx, (images, labels) in enumerate(pbar):
    images, labels = images.to(device), labels.to(device)
    
    logits = model(images)
    loss = criterion(logits, labels)
    loss.backward()
    
    optimizer.step()
    optimizer.zero_grad()
    
    total_loss += loss.item()
    preds = logits.argmax(dim=-1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)
    
    if (batch_idx + 1) % 20 == 0:
        acc = correct / total * 100
        pbar.set_postfix({'loss': f'{total_loss/(batch_idx+1):.4f}', 'acc': f'{acc:.2f}%'})

elapsed = time.time() - start
print(f'Epoch done in {elapsed:.1f}s')
print(f'Train loss: {total_loss/len(train_loader):.4f}, acc: {correct/total*100:.2f}%')

Training:   0%|          | 0/5000 [00:00<?, ?it/s]

d:\Research\ai-image-detection-research\.venv\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch done in 2971.2s
Train loss: 0.6879, acc: 55.70%


In [6]:
# Cell 6: Validate
model.eval()
val_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Validating'):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        val_loss += loss.item()
        preds = logits.argmax(dim=-1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

print(f'Val loss: {val_loss/len(val_loader):.4f}, acc: {val_correct/val_total*100:.2f}%')

Validating:   0%|          | 0/625 [00:00<?, ?it/s]

Val loss: 0.6922, acc: 52.50%


In [7]:
# Cell 7: Full Training Loop (uses model from Cell 4)
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

# Set to 1 for smoke test, change to cfg.training.epochs for real training
NUM_EPOCHS = 1  # cfg.training.epochs

# Track training history for Figure 2
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

# Reuse model from Cell 4 (or rebuild if this cell is run standalone)
try:
    model.to(device)
except NameError:
    from src.model import build_mfft
    model = build_mfft('base').to(device)

warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS * len(train_loader), T_mult=2, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

best_acc = 0
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': f'{total_loss/(total/cfg.training.batch_size):.4f}', 'acc': f'{correct/total*100:.2f}%', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
    
    train_acc = correct / total * 100
    history["train_acc"].append(train_acc)
    history["train_loss"].append(total_loss / len(train_loader))
    
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total * 100
    history["val_acc"].append(val_acc)
    history["val_loss"].append(val_loss / len(val_loader))
    print(f'Epoch {epoch+1}: train={train_acc:.2f}%, val={val_acc:.2f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), PROJECT_ROOT / 'model' / 'checkpoints' / 'best_mfft_base.pt')
        print(f'  Saved best model ({best_acc:.2f}%)')

print(f'\nBest val accuracy: {best_acc:.2f}%')


Epoch 1/50:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 1: train=54.51%, val=61.90%
  Saved best model (61.90%)


Epoch 2/50:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 2: train=59.03%, val=62.68%
  Saved best model (62.68%)


Epoch 3/50:   0%|          | 0/5000 [00:00<?, ?it/s]

...

Epoch 48: train=67.56%, val=68.64%
  Saved best model (68.64%)


Epoch 49/50:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 49: train=67.57%, val=68.14%


Epoch 50/50:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch 50: train=67.79%, val=68.44%

Best val accuracy: 68.64%


In [8]:
# Cell 8: Save Final Model
model.eval()
torch.save(model.state_dict(), PROJECT_ROOT / 'model' / 'checkpoints' / 'mfft_base_final.pt')
print('Model saved to model/checkpoints/mfft_base_final.pt')
print(f'File size: {os.path.getsize(PROJECT_ROOT / "model" / "checkpoints" / "mfft_base_final.pt") / 1e6:.1f} MB')

Model saved to model/checkpoints/mfft_base_final.pt
File size: 3.5 MB


In [9]:
# Cell 9: Test Prediction on a Sample (from test set)
img_path = test_dataset.samples[0][0]
img = Image.open(img_path).convert('RGB')
transform = ImageTransform(size=cfg.training.image_size, augment=False)
tensor = transform(img).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    logits = model(tensor)
    probs = F.softmax(logits, dim=-1)

pred = 'AI-generated' if probs[0][1] > probs[0][0] else 'Real'
print(f'Image: {os.path.basename(img_path)}')
print(f'Real prob: {probs[0][0]:.4f}')
print(f'AI prob:   {probs[0][1]:.4f}')
print(f'Prediction: {pred}')

Image: PEXELS_2909096_aug_crop_95.jpg
Real prob: 0.5994
AI prob:   0.4006
Prediction: Real


---
## Manuscript Figures (Cell 10)
Generate all publication-quality figures after training.

In [10]:
# Cell 10: Generate All Manuscript Figures (on held-out test set)
from src.visualize import generate_all_figures
from sklearn.metrics import confusion_matrix
import numpy as np

FIGS_DIR = PROJECT_ROOT / 'paper' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# Evaluate on held-out test set for paper results
all_test_labels = []
all_test_probs = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=-1)
        all_test_labels.extend(labels.cpu().numpy())
        all_test_probs.extend(probs[:, 1].cpu().numpy())

y_true = np.array(all_test_labels)
y_score = np.array(all_test_probs)
y_pred = (y_score >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)

# Class labels from dataset
all_labels = [s[1] for s in full_dataset.samples]

# Pick a sample image for frequency decomposition
sample_img = None
for cls_dir in ['real', 'BigGAN', 'genimage_ai', 'DALL-E3']:
    d = PROJECT_ROOT / 'dataset' / 'images' / cls_dir
    if d.exists():
        files = list(d.rglob('*.jpg')) or list(d.rglob('*.png'))
        if files:
            sample_img = str(files[0])
            break

# Generate all 10 figures (use test_loader for sample predictions + heatmaps)
paths = generate_all_figures(
    history=history,
    train_loader=train_loader,
    model=model,
    val_loader=test_loader,
    device=device,
    y_true=y_true,
    y_score=y_score,
    cm=cm,
    labels=all_labels,
    sample_image_path=sample_img,
    output_dir=FIGS_DIR,
)

print('\n' + '='*60)
print('All manuscript figures saved to paper/figures/')
for name, path in paths.items():
    print(f'  {name}: {path.name}')
print('='*60)

Generating Figure 1: Frequency Decomposition...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig1_frequency_decomposition.png
Generating Figure 2: Training History...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig2_training_history.png
Generating Figure 3: Confusion Matrix...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig3_confusion_matrix.png
Generating Figure 4: ROC Curve...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig4_roc_curve.png
Generating Figure 5: Precision-Recall Curve...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig5_pr_curve.png
Generating Figure 6: Sample Predictions...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig6_sample_predictions.png
Generating Figure 7: Anomaly Heatmaps...
Saved: d:\Research\ai-image-detection-research\paper\figures\fig7_anomaly_heatmaps.png
Generating Figure 8: Class Distribution...
Saved: d:\Research\ai-image-detection-research\paper\figure

In [11]:
# Cell 11: Per-Category Accuracy Breakdown
from collections import defaultdict
from pathlib import Path
import pandas as pd

# Map image paths to source categories using metadata
meta_map = {}
meta_df = pd.read_csv(PROJECT_ROOT / 'dataset' / 'metadata' / 'clean_metadata.csv', dtype={'generator': str, 'md5': str})
for _, row in meta_df.iterrows():
    fp = str(PROJECT_ROOT / 'dataset' / 'images' / row['filename'])
    meta_map[fp] = row['source']

category_map = defaultdict(list)
for idx, (img_path, true_label) in enumerate(test_dataset.samples):
    source = meta_map.get(img_path, 'unknown')
    if source in ('pexels_unsplash', 'imagenet', 'places365', 'open_images_v7'):
        category = "Real"
    elif source in ('genimage_biggan', 'biggan', 'glide', 'stable_diffusion', 'dalle3', 'midjourney'):
        category = "AI Generated"
    elif source in ('celebdf', 'faceforensics', 'dfdc'):
        category = "AI Altered"
    else:
        category = "Unknown"
    category_map[category].append((y_pred[idx] == true_label, y_score[idx], true_label, y_pred[idx]))

print("=" * 65)
print(f"{'Category':<20} {'Count':>8} {'Accuracy':>10} {'Avg Conf':>10} {'AUC':>8}")
print("-" * 65)
from sklearn.metrics import roc_auc_score
overall_correct = 0
overall_total = 0
rows = []
for cat in ["Real", "AI Generated", "AI Altered", "Unknown"]:
    if cat not in category_map:
        continue
    items = category_map[cat]
    correct_list = [c for c, _, _, _ in items]
    scores = [s for _, s, _, _ in items]
    true_vs_pred = [(t, p) for _, _, t, p in items]
    n = len(correct_list)
    acc = sum(correct_list) / n * 100
    avg_conf = sum(scores) / n * 100
    try:
        auc = roc_auc_score([t for t, _ in true_vs_pred], [p for _, p in true_vs_pred])
    except Exception:
        auc = 0.0
    rows.append((cat, n, acc, avg_conf, auc))
    overall_correct += sum(correct_list)
    overall_total += n
    print(f"{cat:<20} {n:>8} {acc:>9.2f}% {avg_conf:>9.2f}% {auc:>7.4f}")

print("-" * 65)
overall_acc = overall_correct / overall_total * 100
print(f"{'OVERALL':<20} {overall_total:>8} {overall_acc:>9.2f}%")
print("=" * 65)

# Save to table
import json
tables_dir = PROJECT_ROOT / "paper" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
with open(tables_dir / "per_category_accuracy.json", "w") as f:
    import numpy as np
    def _to_py(x):
        return float(x) if isinstance(x, (np.floating, np.integer)) else x
    json.dump({
        "categories": {r[0]: {"count": int(r[1]), "accuracy": round(float(r[2]), 2), "avg_confidence": round(float(r[3]), 2), "auc": round(float(r[4]), 4)} for r in rows},
        "overall": {"count": int(overall_total), "accuracy": round(float(overall_acc), 2)}
    }, f, indent=2, default=_to_py)
print(f"Saved to {tables_dir / 'per_category_accuracy.json'}")


Category                Count   Accuracy   Avg Conf      AUC
-----------------------------------------------------------------
Real                     2500     91.80%     38.63%     nan
AI Generated              274     55.84%     58.48%     nan
AI Altered               2226     42.50%     58.07%     nan
-----------------------------------------------------------------
OVERALL                  5000     67.88%
Saved to d:\Research\ai-image-detection-research\paper\tables\per_category_accuracy.json


d:\Research\ai-image-detection-research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\Research\ai-image-detection-research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\Research\ai-image-detection-research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [12]:
# Cell 12: Save All Metrics as JSON + CSV Tables for Manuscript
import csv, json
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.calibration import calibration_curve

tables_dir = PROJECT_ROOT / 'paper' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)

def _to_serializable(obj):
    if isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

def save_csv(filename, headers, rows):
    p = tables_dir / filename
    with open(p, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(headers)
        w.writerows(rows)
    print(f'  Saved {p.name}')

def save_json(filename, data):
    p = tables_dir / filename
    with open(p, 'w') as f:
        json.dump(data, f, indent=2, default=_to_serializable)
    print(f'  Saved {p.name}')

# ── Table 1: Dataset Statistics ──
if full_dataset is not None:
    all_lbls = [s[1] for s in full_dataset.samples]
    n_real = all_lbls.count(0)
    n_ai = all_lbls.count(1)
    ds_stats = {
        'total': len(all_lbls), 'real': n_real, 'ai_generated': n_ai,
        'train': len(train_dataset), 'val': len(val_dataset), 'test': len(test_dataset),
        'train_pct': round(len(train_dataset)/len(all_lbls)*100, 1),
        'val_pct': round(len(val_dataset)/len(all_lbls)*100, 1),
        'test_pct': round(len(test_dataset)/len(all_lbls)*100, 1),
    }
    save_json('table1_dataset_statistics.json', ds_stats)
    save_csv('table1_dataset_statistics.csv', ['Split', 'Total', 'Real', 'AI', 'Percentage'],
             [['Train', len(train_dataset), '-', '-', f'{ds_stats["train_pct"]}%'],
              ['Validation', len(val_dataset), '-', '-', f'{ds_stats["val_pct"]}%'],
              ['Test', len(test_dataset), '-', '-', f'{ds_stats["test_pct"]}%'],
              ['Full Dataset', len(all_lbls), n_real, n_ai, '100%']])

# ── Table 2: Training History ──
if 'history' in dir() and len(history['train_loss']) > 0:
    save_json('table2_training_history.json', history)
    save_csv('table2_training_history.csv',
             ['Epoch', 'Train Loss', 'Val Loss', 'Train Acc (%)', 'Val Acc (%)'],
             [[i+1, round(history['train_loss'][i], 4), round(history['val_loss'][i], 4),
               round(history['train_acc'][i], 2), round(history['val_acc'][i], 2)]
              for i in range(len(history['train_loss']))])

# ── Table 3: Test Set Evaluation Metrics ──
if y_true is not None and len(y_true) > 0:
    acc = (y_pred == y_true).mean() * 100
    prec = precision_score(y_true, y_pred) * 100
    rec = recall_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred) * 100
    auc = roc_auc_score(y_true, y_score)
    prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='uniform')
    ece = np.mean(np.abs(prob_true - prob_pred))
    metrics = {
        'accuracy_pct': round(acc, 2),
        'precision_pct': round(prec, 2),
        'recall_pct': round(rec, 2),
        'f1_score_pct': round(f1, 2),
        'auc_roc': round(auc, 4),
        'expected_calibration_error': round(ece, 4),
    }
    save_json('table3_evaluation_metrics.json', metrics)
    save_csv('table3_evaluation_metrics.csv', ['Metric', 'Value'],
             [['Accuracy (%)', f'{acc:.2f}'], ['Precision (%)', f'{prec:.2f}'],
              ['Recall (%)', f'{rec:.2f}'], ['F1 Score (%)', f'{f1:.2f}'],
              ['AUC-ROC', f'{auc:.4f}'], ['ECE', f'{ece:.4f}']])

# ── Table 4: Confusion Matrix ──
if cm is not None and cm.size == 4:
    cm_tbl = {'tn': int(cm[0,0]), 'fp': int(cm[0,1]), 'fn': int(cm[1,0]), 'tp': int(cm[1,1])}
    save_json('table4_confusion_matrix.json', cm_tbl)
    save_csv('table4_confusion_matrix.csv', ['', 'Predicted Real', 'Predicted AI'],
             [['Actual Real', cm_tbl['tn'], cm_tbl['fp']], ['Actual AI', cm_tbl['fn'], cm_tbl['tp']]])

# ── Table 5: Per-Category Accuracy ──
if 'category_map' in dir() and category_map:
    cat_data = {}
    cat_rows = []
    for cat in ['Real', 'AI Generated', 'AI Altered']:
        if cat in category_map:
            items = category_map[cat]
            correct_list = [c for c, _, _, _ in items]
            scores = [s for _, s, _, _ in items]
            n = len(correct_list)
            acc = sum(correct_list) / n * 100
            cat_data[cat] = {'count': n, 'accuracy_pct': round(acc, 2), 'avg_confidence_pct': round(sum(scores)/n*100, 2)}
            cat_rows.append([cat, n, f'{acc:.2f}%', f'{sum(scores)/n*100:.2f}%'])
    save_json('table5_per_category_accuracy.json', cat_data)
    save_csv('table5_per_category_accuracy.csv', ['Category', 'Count', 'Accuracy', 'Avg Confidence'], cat_rows)

# ── Table 6: Model Architecture ──
if model is not None:
    model_info = {
        'architecture': 'Multi-Frequency Fusion Transformer (MFFT)',
        'variant': 'base',
        'total_params': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        'image_size': '224x224',
    }
    import os
    ckpt = PROJECT_ROOT / 'model' / 'checkpoints' / 'best_mfft_base.pt'
    if ckpt.exists():
        model_info['checkpoint_size_mb'] = round(ckpt.stat().st_size / 1e6, 2)
    save_json('table6_model_architecture.json', model_info)
    save_csv('table6_model_architecture.csv', ['Property', 'Value'],
             [[k.replace('_', ' ').title(), str(v)] for k, v in model_info.items()])

# ── Table 7: Training Performance ──
if 'history' in dir() and len(history['train_loss']) > 0:
    train_perf = {
        'total_epochs': len(history['train_loss']),
        'best_val_acc_pct': round(max(history['val_acc']), 2),
        'best_val_loss': round(min(history['val_loss']), 4),
        'final_train_acc_pct': round(history['train_acc'][-1], 2),
        'final_val_acc_pct': round(history['val_acc'][-1], 2),
        'final_train_loss': round(history['train_loss'][-1], 4),
        'final_val_loss': round(history['val_loss'][-1], 4),
    }
    save_json('table7_training_performance.json', train_perf)
    save_csv('table7_training_performance.csv', ['Metric', 'Value'],
             [[k.replace('_', ' ').title(), str(v)] for k, v in train_perf.items()])

# ── Table 8: Per-Class Metrics ──
if y_true is not None and len(y_true) > 0:
    from sklearn.metrics import classification_report
    report = classification_report(y_true, y_pred, target_names=['Real', 'AI-Generated'], output_dict=True, zero_division=0)
    per_class = {
        'real': {'precision_pct': round(report['Real']['precision']*100, 2), 'recall_pct': round(report['Real']['recall']*100, 2), 'f1_pct': round(report['Real']['f1-score']*100, 2), 'support': int(report['Real']['support'])},
        'ai_generated': {'precision_pct': round(report['AI-Generated']['precision']*100, 2), 'recall_pct': round(report['AI-Generated']['recall']*100, 2), 'f1_pct': round(report['AI-Generated']['f1-score']*100, 2), 'support': int(report['AI-Generated']['support'])},
    }
    save_json('table8_per_class_metrics.json', per_class)
    save_csv('table8_per_class_metrics.csv', ['Class', 'Precision (%)', 'Recall (%)', 'F1 Score (%)', 'Support'],
             [['Real', per_class['real']['precision_pct'], per_class['real']['recall_pct'], per_class['real']['f1_pct'], per_class['real']['support']],
              ['AI-Generated', per_class['ai_generated']['precision_pct'], per_class['ai_generated']['recall_pct'], per_class['ai_generated']['f1_pct'], per_class['ai_generated']['support']]])

# ── Table 9: Calibration Metrics ──
if y_true is not None and len(y_true) > 0:
    from sklearn.metrics import brier_score_loss
    brier = brier_score_loss(y_true, y_score)
    calib_metrics = {
        'brier_score': round(brier, 4),
        'ece': round(ece, 4),
        'mean_confidence_correct': float(y_score[y_pred == y_true].mean()) if y_pred[y_pred == y_true].size > 0 else 0,
        'mean_confidence_incorrect': float(y_score[y_pred != y_true].mean()) if y_pred[y_pred != y_true].size > 0 else 0,
    }
    save_json('table9_calibration_metrics.json', calib_metrics)
    save_csv('table9_calibration_metrics.csv', ['Metric', 'Value'],
             [[k.replace('_', ' ').title(), str(v)] for k, v in calib_metrics.items()])

print(f'\nAll tables saved to {tables_dir}/')


  Saved table1_dataset_statistics.json
  Saved table1_dataset_statistics.csv
  Saved table2_training_history.json
  Saved table2_training_history.csv
  Saved table3_evaluation_metrics.json
  Saved table3_evaluation_metrics.csv
  Saved table4_confusion_matrix.json
  Saved table4_confusion_matrix.csv
  Saved table5_per_category_accuracy.json
  Saved table5_per_category_accuracy.csv
  Saved table6_model_architecture.json
  Saved table6_model_architecture.csv
  Saved table7_training_performance.json
  Saved table7_training_performance.csv
  Saved table8_per_class_metrics.json
  Saved table8_per_class_metrics.csv
  Saved table9_calibration_metrics.json
  Saved table9_calibration_metrics.csv

All tables saved to d:\Research\ai-image-detection-research\paper\tables/
